# PoC 2: Evoked-Response Anomaly Detection

**Question:** Can we flag recordings where evoked responses have drifted from baseline?

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from _common import load_db, load_evoked_feature_matrix
plt.style.use('dark_background')
DB_PATH = os.path.join('..', 'data', 'monitor.db')

## 1. Load evoked feature matrix

In [ ]:
df = load_evoked_feature_matrix(DB_PATH)
print(f"Shape: {df.shape}")
print(f"Sessions: {df['session_dir'].nunique()}")
print(f"Date range: {df['chunk_datetime'].min()} -- {df['chunk_datetime'].max()}")
print()

# All numeric evoked-response features
FEATURE_COLS = [
    'line_length', 'log_auc', 'peak_amplitude', 'trough_amplitude',
    'peak_to_trough', 'rms_amplitude', 'peak_latency_ms', 'trough_latency_ms',
    'max_slope', 'max_slope_time_ms', 'early_area', 'late_area',
    'early_late_ratio', 'recovery_tau', 'recovery_slope',
    'template_correlation', 'pca_recon_error', 'variance',
    'autocorrelation', 'ac_width', 'exp_fit_a', 'sum_power_low',
    'freq_moment_low', 'sum_power_high', 'freq_moment_high',
]
assert len(FEATURE_COLS) >= 25, f"Expected >=25 features but got {len(FEATURE_COLS)}"

# Verify all columns exist in the DataFrame
missing = [c for c in FEATURE_COLS if c not in df.columns]
assert len(missing) == 0, f"Missing feature columns: {missing}"
print(f"Using {len(FEATURE_COLS)} numeric features")
print()
print(df[FEATURE_COLS].describe().T.to_string())
print()

# Drop artifact rows
n_before = len(df)
df = df[df['is_artifact'] != 1].reset_index(drop=True)
print(f"Dropped {n_before - len(df)} artifact rows, {len(df)} remaining")

## 2. Feature exploration

In [ ]:
# Histogram grid of all features
n_feats = len(FEATURE_COLS)
ncols = 5
nrows = int(np.ceil(n_feats / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
axes_flat = axes.flatten()

for idx in range(len(axes_flat)):
    ax = axes_flat[idx]
    if idx < n_feats:
        col = FEATURE_COLS[idx]
        vals = df[col].dropna()
        ax.hist(vals, bins=50, alpha=0.8, edgecolor='none')
        ax.set_title(col, fontsize=9)
        ax.tick_params(labelsize=7)
    else:
        ax.set_visible(False)

fig.suptitle('Feature Distributions (artifact-free)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Correlation heatmap
corr = df[FEATURE_COLS].corr()
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, annot=False, cmap='coolwarm', center=0, vmin=-1, vmax=1,
            xticklabels=True, yticklabels=True, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

# Identify highly correlated pairs (|r| > 0.9)
high_corr_pairs = []
for i in range(len(FEATURE_COLS)):
    for j in range(i + 1, len(FEATURE_COLS)):
        r = corr.iloc[i, j]
        if abs(r) > 0.9:
            high_corr_pairs.append((FEATURE_COLS[i], FEATURE_COLS[j], round(r, 3)))

print(f"Highly correlated pairs (|r| > 0.9): {len(high_corr_pairs)}")
for a, b, r in sorted(high_corr_pairs, key=lambda x: -abs(x[2])):
    print(f"  {a:30s} <-> {b:30s}  r={r:+.3f}")

## 3. Per-mouse baseline with IsolationForest

In [ ]:
scored_parts = []
sessions = df['session_dir'].unique()
print(f"Fitting IsolationForest per session ({len(sessions)} sessions)...")

for session in sessions:
    sdf = df[df['session_dir'] == session].sort_values('chunk_datetime').copy()
    if len(sdf) < 10:
        # Too few epochs to fit a meaningful model
        sdf['if_score'] = np.nan
        sdf['if_label'] = 0
        scored_parts.append(sdf)
        continue

    # Replace NaN features with column medians for this session
    feat_df = sdf[FEATURE_COLS].copy()
    feat_df = feat_df.fillna(feat_df.median())

    # First 25% of epochs (by time) as baseline
    n_baseline = max(5, int(len(feat_df) * 0.25))
    X_baseline = feat_df.iloc[:n_baseline].values
    X_all = feat_df.values

    # Scale features using baseline statistics
    scaler = StandardScaler()
    scaler.fit(X_baseline)
    X_baseline_sc = scaler.transform(X_baseline)
    X_all_sc = scaler.transform(X_all)

    # Handle any remaining NaN/inf after scaling
    X_baseline_sc = np.nan_to_num(X_baseline_sc, nan=0.0, posinf=0.0, neginf=0.0)
    X_all_sc = np.nan_to_num(X_all_sc, nan=0.0, posinf=0.0, neginf=0.0)

    iso = IsolationForest(
        n_estimators=200,
        contamination=0.05,
        random_state=42,
    )
    iso.fit(X_baseline_sc)

    # score_samples returns negative scores; more negative = more anomalous
    sdf['if_score'] = iso.score_samples(X_all_sc)
    sdf['if_label'] = iso.predict(X_all_sc)  # 1=normal, -1=anomaly
    scored_parts.append(sdf)

scored_df = pd.concat(scored_parts, ignore_index=True)
print(f"Scored {len(scored_df)} epochs across {len(sessions)} sessions")
print(f"Anomalies flagged (IF label=-1): {(scored_df['if_label'] == -1).sum()}")

In [ ]:
# Visualize anomaly scores
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Time series per session (scatter, color by score)
ax = axes[0]
valid = scored_df.dropna(subset=['if_score'])
sc = ax.scatter(
    pd.to_datetime(valid['chunk_datetime']),
    valid['if_score'],
    c=valid['if_score'],
    cmap='coolwarm_r',
    s=3,
    alpha=0.5,
)
ax.set_xlabel('Date')
ax.set_ylabel('IF Anomaly Score')
ax.set_title('Anomaly Score Over Time')
plt.colorbar(sc, ax=ax, label='Score')

# Distribution of scores
ax = axes[1]
ax.hist(valid['if_score'], bins=80, alpha=0.8, edgecolor='none')
ax.axvline(valid['if_score'].quantile(0.10), color='red', ls='--',
           label='10th percentile')
ax.set_xlabel('IF Anomaly Score')
ax.set_ylabel('Count')
ax.set_title('Score Distribution')
ax.legend()

plt.tight_layout()
plt.show()

# Top 20 most anomalous files by mean score
file_scores = (
    scored_df.dropna(subset=['if_score'])
    .groupby('file_id')
    .agg(
        session=('session_dir', 'first'),
        datetime=('chunk_datetime', 'first'),
        mean_score=('if_score', 'mean'),
        n_epochs=('if_score', 'count'),
    )
    .sort_values('mean_score')
)
print("Top 20 most anomalous files (lowest mean IF score):")
print(file_scores.head(20).to_string())

## 4. Autoencoder comparison

In [ ]:
class EvokedAutoencoder(nn.Module):
    """Shallow autoencoder: 26 -> 8 -> 26."""
    def __init__(self, input_dim=26, hidden_dim=8):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, hidden_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, 16),
            nn.ReLU(),
            nn.Linear(16, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

In [ ]:
# Prepare data: same per-session baseline split, scaled globally
baseline_mask = np.zeros(len(scored_df), dtype=bool)
for session in scored_df['session_dir'].unique():
    idx = scored_df[scored_df['session_dir'] == session].index
    if len(idx) < 10:
        continue
    sdf_sorted = scored_df.loc[idx].sort_values('chunk_datetime')
    n_bl = max(5, int(len(sdf_sorted) * 0.25))
    baseline_mask[sdf_sorted.index[:n_bl]] = True

feat_all = scored_df[FEATURE_COLS].fillna(0.0).values
feat_all = np.nan_to_num(feat_all, nan=0.0, posinf=0.0, neginf=0.0)

scaler_ae = StandardScaler()
scaler_ae.fit(feat_all[baseline_mask])
X_scaled = scaler_ae.transform(feat_all).astype(np.float32)
X_scaled = np.nan_to_num(X_scaled, nan=0.0, posinf=0.0, neginf=0.0)

X_train = X_scaled[baseline_mask]
X_all_ae = X_scaled

input_dim = X_train.shape[1]
print(f"Input dim: {input_dim}, Training samples: {len(X_train)}, Total: {len(X_all_ae)}")

# DataLoader for training
train_ds = TensorDataset(torch.tensor(X_train))
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

# Train
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = EvokedAutoencoder(input_dim=input_dim, hidden_dim=8).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

n_epochs = 100
losses = np.zeros(n_epochs)
for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0.0
    n_batches = 0
    for (batch_x,) in train_loader:
        batch_x = batch_x.to(device)
        recon, _ = model(batch_x)
        loss = criterion(recon, batch_x)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    losses[epoch] = epoch_loss / max(n_batches, 1)

# Plot training loss
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(losses, linewidth=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Autoencoder Training Loss')
plt.tight_layout()
plt.show()
print(f"Final training loss: {losses[-1]:.6f}")

# Compute reconstruction error on all data
model.eval()
with torch.no_grad():
    X_tensor = torch.tensor(X_all_ae).to(device)
    recon, latent = model(X_tensor)
    recon_err = ((recon - X_tensor) ** 2).mean(dim=1).cpu().numpy()

scored_df['ae_recon_error'] = recon_err
print(f"Recon error -- mean: {recon_err.mean():.4f}, "
      f"median: {np.median(recon_err):.4f}, "
      f"95th: {np.percentile(recon_err, 95):.4f}")

In [ ]:
from scipy.stats import spearmanr

# Compare IF score vs autoencoder reconstruction error
valid_mask = scored_df['if_score'].notna() & scored_df['ae_recon_error'].notna()
cmp = scored_df[valid_mask].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: IF score vs AE recon error
ax = axes[0]
ax.scatter(cmp['if_score'], cmp['ae_recon_error'], s=2, alpha=0.3)
ax.set_xlabel('Isolation Forest Score')
ax.set_ylabel('Autoencoder Recon Error')
ax.set_title('IF Score vs AE Reconstruction Error')

# Rank correlation
rho, pval = spearmanr(cmp['if_score'], cmp['ae_recon_error'])
ax.text(0.05, 0.95, f"Spearman rho = {rho:.3f}\np = {pval:.2e}",
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='black', alpha=0.5))

# Venn-style overlap of top-10% flagged by each method
n_top = max(1, int(len(cmp) * 0.10))
top_if = set(cmp.nsmallest(n_top, 'if_score').index)  # lowest IF = most anomalous
top_ae = set(cmp.nlargest(n_top, 'ae_recon_error').index)  # highest AE = most anomalous
both = top_if & top_ae
only_if = top_if - top_ae
only_ae = top_ae - top_if

ax = axes[1]
labels = ['IF only', 'Both', 'AE only']
counts = [len(only_if), len(both), len(only_ae)]
colors = ['#4c72b0', '#55a868', '#c44e52']
bars = ax.bar(labels, counts, color=colors, edgecolor='white')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            str(count), ha='center', va='bottom', fontsize=11)
ax.set_ylabel('Number of Epochs')
ax.set_title(f'Top-10% Anomaly Overlap (n={n_top} each)')

plt.tight_layout()
plt.show()

print(f"Spearman rho: {rho:.3f} (p={pval:.2e})")
print(f"Top-10% overlap: {len(both)} of {n_top} ({len(both)/n_top:.1%})")

## 5. UMAP visualization

In [ ]:
import umap

# Subsample if >50k points for UMAP tractability
max_umap_pts = 50000
if len(scored_df) > max_umap_pts:
    umap_idx = np.random.RandomState(42).choice(len(scored_df), max_umap_pts, replace=False)
    umap_df = scored_df.iloc[umap_idx].copy()
else:
    umap_df = scored_df.copy()
    umap_idx = np.arange(len(scored_df))

X_umap = X_scaled[umap_idx]
X_umap = np.nan_to_num(X_umap, nan=0.0, posinf=0.0, neginf=0.0)

reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1, random_state=42)
embedding = reducer.fit_transform(X_umap)

umap_df = umap_df.reset_index(drop=True)
umap_df['umap_0'] = embedding[:, 0]
umap_df['umap_1'] = embedding[:, 1]

# 3 subplots: color by session, anomaly score, time
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# (a) Color by session_dir
ax = axes[0]
session_codes = pd.Categorical(umap_df['session_dir']).codes
sc = ax.scatter(umap_df['umap_0'], umap_df['umap_1'],
                c=session_codes, cmap='tab20', s=3, alpha=0.5)
ax.set_title('Colored by Session')
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')

# (b) Color by anomaly score
ax = axes[1]
if_scores = umap_df['if_score'].fillna(0).values
sc = ax.scatter(umap_df['umap_0'], umap_df['umap_1'],
                c=if_scores, cmap='coolwarm_r', s=3, alpha=0.5)
ax.set_title('Colored by IF Anomaly Score')
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
plt.colorbar(sc, ax=ax, label='IF Score')

# (c) Color by time
ax = axes[2]
time_vals = pd.to_datetime(umap_df['chunk_datetime'])
time_numeric = (time_vals - time_vals.min()).dt.total_seconds()
sc = ax.scatter(umap_df['umap_0'], umap_df['umap_1'],
                c=time_numeric, cmap='viridis', s=3, alpha=0.5)
ax.set_title('Colored by Time')
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
plt.colorbar(sc, ax=ax, label='Seconds from start')

plt.tight_layout()
plt.show()

## 6. Validation against known issues

In [ ]:
conn = load_db(DB_PATH)
try:
    alerts_df = pd.read_sql_query(
        "SELECT alert_type, severity, message, file_id, session_dir, sent_at FROM alerts",
        conn,
    )
    annotations_df = pd.read_sql_query(
        "SELECT timestamp, session_dir, file_id, note, category FROM annotations",
        conn,
    )
finally:
    conn.close()

print(f"Alerts: {len(alerts_df)} rows")
print(f"Annotations: {len(annotations_df)} rows")

# Combine known-bad file_ids from both sources
alert_file_ids = set(alerts_df['file_id'].dropna().astype(int).tolist())
annot_file_ids = set(annotations_df['file_id'].dropna().astype(int).tolist())
known_bad_ids = alert_file_ids | annot_file_ids
print(f"Unique known-bad file_ids: {len(known_bad_ids)}")

# Check which known-bad files are in top-10% anomaly scores
file_scores = (
    scored_df.dropna(subset=['if_score'])
    .groupby('file_id')['if_score']
    .mean()
)
threshold_10pct = file_scores.quantile(0.10)  # lowest 10% are most anomalous
flagged_file_ids = set(file_scores[file_scores <= threshold_10pct].index.tolist())

# Compute recall
known_in_scored = known_bad_ids & set(file_scores.index.tolist())
if len(known_in_scored) > 0:
    caught = known_in_scored & flagged_file_ids
    recall_known = len(caught) / len(known_in_scored)
else:
    recall_known = 0.0

print(f"\nKnown-bad files present in scored data: {len(known_in_scored)}")
print(f"Caught in top-10% anomaly scores: {len(caught) if known_in_scored else 0}")
print(f"Recall of known issues: {recall_known:.2%}")

# Break down by alert type
if len(alerts_df) > 0:
    print("\nAlert type breakdown:")
    for atype in alerts_df['alert_type'].unique():
        atype_ids = set(
            alerts_df[alerts_df['alert_type'] == atype]['file_id']
            .dropna().astype(int).tolist()
        )
        atype_in_scored = atype_ids & set(file_scores.index.tolist())
        if len(atype_in_scored) > 0:
            atype_caught = atype_in_scored & flagged_file_ids
            r = len(atype_caught) / len(atype_in_scored)
            print(f"  {atype}: {len(atype_caught)}/{len(atype_in_scored)} = {r:.0%}")

# Annotation categories
if len(annotations_df) > 0:
    print("\nAnnotation categories:")
    print(annotations_df['category'].value_counts().to_string())

## 7. Verdict

In [ ]:
# recall_known = fraction of known-bad sessions in top 10% anomaly scores
if recall_known >= 0.70:
    verdict = "SHIP"
    reason = f"Catches {recall_known:.0%} of known issues in top 10% scores"
elif recall_known >= 0.50:
    verdict = "ITERATE"
    reason = f"Catches {recall_known:.0%} -- promising but needs feature tuning"
else:
    verdict = "DROP"
    reason = f"Only catches {recall_known:.0%} -- thresholds are sufficient"
print(f"VERDICT: {verdict}")
print(f"Reason: {reason}")